# 2. Probes

Six families. Five go after a protected field. The sixth never mentions it, and that one
carries the entire detectability result.

In [ ]:
# On Kaggle or Colab, uncomment to install
# !pip install -q -e /kaggle/working/silentwall
# !pip install -q -e .

from silentwall.config import load_config
from silentwall.pipeline import prepare_workspace, run_method, run_sweep, save_workspace
from silentwall.report.render import render_comparison, render_markdown, write_comparison

CONFIG = "../configs/smoke.yaml"
cfg = load_config(CONFIG)
print(cfg.profile, cfg.tier, "methods:", len(cfg.methods))

In [ ]:
ws = prepare_workspace(cfg, verbose=False)

from collections import Counter
counts = Counter(p.family.value for p in ws.probes)
for family, n in sorted(counts.items()):
    print(f"{family:20s} {n}")

## Content probes

These target a specific protected field, so they only go to restricted entities. A control
entity has nothing to leak.

In [ ]:
for p in ws.probes:
    if not p.is_behavioural:
        print(f"[{p.family.value}] {p.prompt}")
        if p.family.value == "inference_chain":
            break

## Behavioural probes

These are the interesting ones. Ordinary questions, nothing confidential, answerable by an
agent that never held the protected information. The identical set goes to restricted and
control entities.

That symmetry is what makes the comparison valid. If restricted and control entities got
different questions, a classifier separating them would be reading the question rather
than the answer.

In [ ]:
restricted = ws.corpus.restricted[0]
control = ws.corpus.controls[0]

for entity in (restricted, control):
    print(f"--- {entity.entity_class}: {entity.display_name} ---")
    shown = 0
    for p in ws.probes:
        if p.entity_id == entity.entity_id and p.is_behavioural:
            print(" ", p.prompt)
            shown += 1
            if shown == 4:
                break
    print()